# autoresearch-medimage - Colab benchmark demo

This notebook runs one ChestX-ray14 baseline experiment from a dataset copy you already have.

Use this notebook when you want:
- one end-to-end ChestXray14 benchmark demo run
- a saved checkpoint
- paper and figure artifacts from that single run

If you want an actual multi-experiment search session with `--session`, keep/discard decisions, and Drive sync, use `notebooks/colab_autoresearch.ipynb` instead.

## 1. Setup

In [ ]:
# Check GPU availability
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Install uv
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]

In [ ]:
# Download the public repo snapshot into /content
!rm -rf /content/autoresearch-medimage /content/autoresearch-medimage-main /content/repo.zip
!wget -q https://github.com/mlungrengmail/autoresearch-medimage/archive/refs/heads/main.zip -O /content/repo.zip
!unzip -q /content/repo.zip -d /content
!mv /content/autoresearch-medimage-main /content/autoresearch-medimage
%cd /content/autoresearch-medimage

In [ ]:
# Install dependencies
!uv sync

## 2. Point the repo at your ChestX-ray14 copy

Recommended source paths:
- Google Drive folder such as `/content/drive/MyDrive/chestxray14`
- Google Drive archive such as `/content/drive/MyDrive/chestxray14.zip`

For a single baseline run, the default is to symlink the Drive folder directly into the repo.
That keeps setup fast. For repeated runs, use the autoresearch notebook and local SSD caching.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# Examples:
# CHESTXRAY14_SOURCE = '/content/drive/MyDrive/chestxray14'
# CHESTXRAY14_SOURCE = '/content/drive/MyDrive/chestxray14.zip'
CHESTXRAY14_SOURCE = ''
USE_LOCAL_SSD = False

In [ ]:
import os
import shutil
import tarfile
import zipfile
from pathlib import Path

source = Path(CHESTXRAY14_SOURCE)
repo_data = Path('/content/autoresearch-medimage/data/chestxray14')
cache_dir = Path('/content/chestxray14_cache')

if not str(source).strip():
    raise ValueError('Set CHESTXRAY14_SOURCE before running this cell.')
if not source.exists():
    raise FileNotFoundError(source)

if repo_data.is_symlink() or repo_data.exists():
    if repo_data.is_symlink():
        repo_data.unlink()
    elif repo_data.is_dir():
        shutil.rmtree(repo_data)
    else:
        repo_data.unlink()

target = source
if USE_LOCAL_SSD or source.is_file():
    if cache_dir.exists():
        shutil.rmtree(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    if source.is_dir():
        shutil.copytree(source, cache_dir, dirs_exist_ok=True)
    elif source.suffix.lower() == '.zip':
        with zipfile.ZipFile(source) as archive:
            archive.extractall(cache_dir)
    elif source.name.endswith(('.tar.gz', '.tgz', '.tar')):
        with tarfile.open(source) as archive:
            archive.extractall(cache_dir)
    else:
        raise ValueError('Use a folder, .zip, .tar.gz, .tgz, or .tar archive.')
    target = cache_dir

os.symlink(target, repo_data, target_is_directory=True)
print('linked:', repo_data)
print('points to:', os.readlink(repo_data))

## 3. Prepare and run one baseline

Use `--refresh-splits` here to rebuild `splits.csv` with Colab/Linux paths, especially if your dataset copy came from Windows.

In [ ]:
!uv run prepare.py --dataset chestxray14 --refresh-splits

In [ ]:
# Run one 300-second baseline experiment
!uv run train.py --recipe chestxray14_baseline --save-checkpoint

In [ ]:
# Finalize the run: held-out test eval, figures, and paper draft
!uv run python tools/finalize_run.py --dataset chestxray14

In [ ]:
# Optional: back up the run outputs to Drive before the runtime resets
!mkdir -p /content/drive/MyDrive/autoresearch_runs/chestxray14_single_run
!cp results.tsv /content/drive/MyDrive/autoresearch_runs/chestxray14_single_run/
!cp progress.png /content/drive/MyDrive/autoresearch_runs/chestxray14_single_run/
!cp -r checkpoints /content/drive/MyDrive/autoresearch_runs/chestxray14_single_run/
!cp -r paper/generated /content/drive/MyDrive/autoresearch_runs/chestxray14_single_run/

In [ ]:
# Inspect the latest ChestX-ray14 ledger rows
import pandas as pd
df = pd.read_csv('results.tsv', sep='\t')
df[df['dataset'] == 'chestxray14'][['timestamp', 'recipe', 'backbone', 'metric', 'status']].tail(10)